[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/solutions/04_capacity_prevalence_queues.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Capacity, prevalence and daily queues

Allow about 35 minutes. Can a policy that fits average capacity still
overwhelm a service? You will project recording-level counts to a new
prevalence, simulate daily demand, and choose a threshold under an explicit
overflow tolerance. Recording labels are real CinC labels; every model score,
and every day of arrivals, is simulated.

The action is **send this recording for AF review**, priced as in the core
notebook: $C_{TP}$ = 1, $C_{FP}$ = 1, $C_{FN}$ = 5 and $C_{TN}$ = 0, in illustrative loss units.
Replace each `...` and run the checker below it; if you are stuck for about
10 minutes, use the stuck switch below the function, where there is one, or open the worked copy in `extension/solutions/`.
A later cell that needs an unfinished problem prints `Waiting for ...`: once that
problem is done or switched, click in the waiting cell, choose **Runtime > Run
before**, then run the waiting cell.

In [ ]:
import numpy as np
from IPython.display import display
from clinical_utility import data, reference, deployment
from clinical_utility import extension_checks as check, extension_plots as view

costs = dict(reference.DEFAULT_COSTS)
capacity = 50            # reviews per day
arrivals_per_day = 1000  # recordings per day
prevalence = .05         # the planning prevalence; Problem F changes it

## Capacity is a daily constraint

A service can review at most 50 recordings for every 1,000 that arrive in a
day. At a prevalence of 5%, the development forest policy with the lowest
projected loss would request 72.5 reviews per 1,000. Restricting the mean
to capacity gives a policy with 49.7. These are expected counts.

### Problem A: Predict

A policy's mean daily demand is below capacity. Can it still exceed capacity
on some days? Store `"never"` or `"sometimes"` before simulating.

In [ ]:
overflow_prediction = "sometimes"
_ = check.run("queue_predict", overflow_prediction)

A constraint on the mean does not promise that every day fits. Arrivals vary,
and so do the AF mix and the decisions. Demand above capacity needs a plan:
extra staff, delayed review, or a capped selection. Each has different
consequences.

## Project the natural frequencies

Hold sensitivity and specificity fixed while the prevalence changes. The AF
group is `population * prevalence` and the rest is its complement. Apply the
sensitivity to the first group and the false-positive rate to the second. With
sensitivity 80%, specificity 90% and prevalence 5% among 1,000
recordings, the expected counts are TP 40, FN 10, FP 95 and TN 855. The
false-positive rate's denominator is the recordings without AF, not all
recordings.

### Problem B: Make

Write `expected_counts(se, sp, prevalence, population)` returning a dictionary
with the keys `TP`, `FP`, `FN` and `TN`. Keep fractional values. Do not rescale
PPV: it changes when the sizes of the two groups change.

In [ ]:
def expected_counts(se, sp, prevalence, population=1000):
    af, non_af = population * prevalence, population * (1 - prevalence)
    return {"TP": af * se, "FN": af * (1 - se), "FP": non_af * (1 - sp), "TN": non_af * sp}
_ = check.run("queue_counts", expected_counts)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# expected_counts = check.use_reference("expected_counts")

### Problem C: Modify

`policy_rows` below holds each cutoff's projected reviews and priced loss per
1,000 arrivals. Write `select_mean_feasible(rows, capacity)`: discard the rows
whose expected reviews exceed capacity, then return the threshold of the
cheapest remaining row, one number. Do not cap a row's review count while
leaving its TP and FN unchanged: that describes an impossible policy.

In [ ]:
def select_mean_feasible(rows, capacity):
    feasible = [row for row in rows if row["reviews"] <= capacity]
    return min(feasible, key=lambda row: row["loss"])["threshold"]
_ = check.run("queue_select", select_mean_feasible)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# select_mean_feasible = check.use_reference("select_mean_feasible")

In [ ]:
sweep = reference.sweep_thresholds(data.load_records("random_forest", "development"), costs)


def project(prevalence):
    """Each development cutoff's expected reviews, misses and loss per day at a prevalence."""
    rows = []
    for row in sweep:
        counts = expected_counts(row["sensitivity"], row["specificity"], prevalence, arrivals_per_day)
        rows.append({"threshold": row["threshold"], "reviews": counts["TP"] + counts["FP"],
                     "misses": counts["FN"], "loss": sum(counts[k] * costs[k] for k in counts),
                     "sensitivity": row["sensitivity"], "specificity": row["specificity"]})
    return rows


def mean_feasible_choice():
    """The Problem C cutoff at today's prevalence, and its projected row."""
    rows = project(prevalence)
    cut = select_mean_feasible(rows, capacity)
    return cut, next(row for row in rows if row["threshold"] == cut)


# mean_cut, the Problem C cutoff, is used again by the top-k demonstration and Problem F.
# The later cells call mean_feasible_choice() again, so each also runs on its own.
if check.ready(expected_counts=expected_counts, select_mean_feasible=select_mean_feasible):
    policy_rows = project(prevalence)
    print("Cheapest overall:", min(policy_rows, key=lambda row: row["loss"]))
    mean_cut, chosen = mean_feasible_choice()
    print("Cheapest within capacity on average:", chosen)

## Simulate days, not just the mean

Draw each day's arrivals from a Poisson distribution with the stated mean.
Given the arrivals, draw the number entering review from a binomial
distribution with a fixed review probability. This model assumes independent
arrivals and a stable review probability. It leaves out weekly patterns,
clustered referrals and backlogs. Its overflow is **new demand above that day's
capacity**, not a waiting list carried over from yesterday.

### Problem D: Make

Implement `fraction_over_capacity`. Draw all the days' arrivals in one call,
`arrivals = rng.poisson(arrival_mean, size=days)`, then all the reviews in one
call with `rng.binomial`. (A loop that draws one day at a time is also
accepted.) Return the fraction of days on which reviews are strictly greater
than capacity. A day exactly at capacity is not overflow.

In [ ]:
def fraction_over_capacity(arrival_mean, review_rate, capacity, days=365, seed=20261106):
    rng = np.random.default_rng(seed)
    arrivals = rng.poisson(arrival_mean, size=days)
    reviews = rng.binomial(arrivals, review_rate)
    return float(np.mean(reviews > capacity))
_ = check.run("queue_simulate", fraction_over_capacity)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# fraction_over_capacity = check.use_reference("fraction_over_capacity")

In [ ]:
if check.ready(expected_counts=expected_counts, select_mean_feasible=select_mean_feasible,
               fraction_over_capacity=fraction_over_capacity):
    mean_cut, chosen = mean_feasible_choice()
    rate = chosen["reviews"] / arrivals_per_day
    print("Fraction of days over capacity:", fraction_over_capacity(arrivals_per_day, rate, capacity))
    display(view.queue(deployment.daily_reviews(arrivals_per_day, rate), capacity))
    print("Poisson calculation of the same probability:",
          deployment.overflow_probability(chosen["reviews"], capacity))

For the 49.7-review policy, the seeded simulation overflows on
47% of days. Its mean fits; its daily demand often does not. The Poisson tail
is a separate calculation under the same model, so the two need not agree
exactly.

## Choose an overflow pathway

A top-k policy ranks each day's recordings and reviews only the highest
scores, breaking ties by arrival order. It caps the workload by construction,
but its effective cutoff moves with each day's scores, and lower-ranked
recordings lose review on busy days. The sensitivity and specificity of a
fixed threshold do not carry over to top-k, so its evaluation must reproduce
the daily batches and the consequences of deferral. The demonstration below
uses one batch of 1,000 development recordings to show the cap against the
Problem C cutoff; it is not an evaluation at the projected prevalence.

In [ ]:
# Change k and see which recordings at or above the Problem C cutoff lose review.
batch_scores = data.load_arrays("development")["random_forest"][:arrivals_per_day]
review_top_k = deployment.top_k_policy(batch_scores, k=capacity)
print("Top-k reviews:", review_top_k.sum())
if check.ready(expected_counts=expected_counts, select_mean_feasible=select_mean_feasible):
    mean_cut, chosen = mean_feasible_choice()
    review_cut = batch_scores >= mean_cut
    print(f"Reviews at the Problem C cutoff, {mean_cut:.3f}:", review_cut.sum())
    print("Of those, recordings that the cap excludes:", np.sum(review_cut & ~review_top_k))

### Problem E: Fix the bug

A planning note chose its policy with `select_policy` below. The service
accepts an overflow probability of at most 10%, but the code still
treats "mean within capacity" as "acceptable daily service". Fix it in place so
that it keeps the rows whose `overflow` is at most `tolerance` and returns the
cheapest one's threshold. The rows use the Poisson overflow probability, so
the choice is not driven by simulation noise.

Then record in `overflow_plan` what happens to recordings on an overflow day.
The tolerance is a service preference supplied for this exercise, not a
statistical result.

In [ ]:
def select_policy(rows, tolerance):
    """The cheapest policy that the service can run day to day."""
    feasible = [row for row in rows if row["overflow"] <= tolerance]
    return min(feasible, key=lambda row: row["loss"])["threshold"]


overflow_plan = """
On a day over capacity the lowest-scoring flagged recordings wait until the next
day's list, and the service reports how many waited and for how long.
"""
_ = check.run("queue_tolerance", select_policy)

In [ ]:
if check.ready(expected_counts=expected_counts, select_policy=select_policy):
    tolerance_rows = project(prevalence)
    for row in tolerance_rows:
        row["overflow"] = deployment.overflow_probability(row["reviews"], capacity)
    selected = select_policy(tolerance_rows, .1)
    print("Policy within the overflow tolerance:",
          next(row for row in tolerance_rows if row["threshold"] == selected))

### Problem F: Modify

Prevalence drifts. Keep the cutoff you chose in Problem C, `mean_cut`, fixed,
with its development sensitivity and specificity. `mean_cut` and `project`
need Problems B and C, or their stuck switches. Use `project` with prevalences from
1% to 10% in steps of 0.5%, and store in `highest_safe_prevalence` the highest
prevalence at which that cutoff's overflow probability is still at most
10%. Store it as a proportion.

In [ ]:
highest_safe_prevalence = None
for k in range(2, 21):
    new_prevalence = k / 200
    row = next(r for r in project(new_prevalence) if r["threshold"] == mean_cut)
    if deployment.overflow_probability(row["reviews"], capacity) <= .1:
        highest_safe_prevalence = new_prevalence
_ = check.run("queue_prevalence", highest_safe_prevalence)

## Before you finish

- Give the AF and non-AF denominators before calculating expected counts.
- Distinguish a constraint on the mean from an overflow tolerance.
- Describe what top-k does to a recording on a busy day.
- Name the prevalence and within-class quantities you would monitor.

**Discuss:** If demand is clustered and yesterday's queue carries over, which
assumptions fail, and what service data would you collect before choosing a
new policy?